In [1]:
# Data handling library
import pandas as pd

# Save and load model file
import joblib

# Split dataset into train and test
from sklearn.model_selection import train_test_split

# Convert text columns into numbers
from sklearn.preprocessing import LabelEncoder

# Accuracy and report
from sklearn.metrics import accuracy_score, classification_report

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [2]:
# Read CSV file
df = pd.read_csv("loan_data.csv")

# Show first 5 rows
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [3]:
# Show dataset information
df.info()

# Check missing values in each column
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  object 
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  object 
 11  EmploymentType  255347 non-null  object 
 12  MaritalStatus   255347 non-null  object 
 13  HasMortgage     255347 non-null  object 
 14  HasDependents   255347 non-null  object 
 15  LoanPurpose     255347 non-null  object 
 16  HasCoSigner     255347 non-null  object 
 17  Default   

LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64

In [4]:
# LoanID is not useful for prediction
if "LoanID" in df.columns:
    df = df.drop("LoanID", axis=1)

# Show updated data
df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [5]:
# Create encoder object
le = LabelEncoder()

# Find text columns and convert to numbers
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# Show converted data
df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,0,0,0,1,1,4,1,0
1,69,50432,124440,458,15,1,4.81,60,0.68,2,0,1,0,0,4,1,0
2,46,84208,129188,451,26,3,21.17,24,0.31,2,3,0,1,1,0,0,1
3,32,31713,44799,743,0,3,7.07,24,0.23,1,0,1,0,0,1,0,0
4,60,20437,9139,633,8,4,6.51,48,0.73,0,3,0,0,1,0,0,0


In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Random Forest": RandomForestClassifier(
        n_estimators=50,
        n_jobs=-1,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(n_estimators=50),

    "XGBoost": XGBClassifier(
        n_estimators=50,
        max_depth=4,
        learning_rate=0.1,
        scale_pos_weight=7,
        random_state=42
    )
}

In [7]:
# Features = only UI input columns
X = df[[
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio"
]]

# Target = output column
y = df["Default"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (255347, 9)
y shape: (255347,)


In [8]:
# 80% training data
# 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Print sizes
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (204277, 9)
Test: (51070, 9)


In [9]:
models = {

    # Better for imbalance
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ),

    # Better for imbalance
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=7,
        random_state=42
    )
}

In [10]:
# Variables to store best model
best_model = None
best_name = ""
best_score = 0

# Loop through all models
for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predict using test data
    pred = model.predict(X_test)

    # Calculate accuracy
    score = accuracy_score(y_test, pred)

    # Print model score
    print(name, ":", round(score, 4))

    # Save if best score found
    if score > best_score:
        best_score = score
        best_model = model
        best_name = name

c:\Users\venka\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression : 0.6723
Random Forest : 0.8853
Gradient Boosting : 0.8868
XGBoost : 0.7155


In [11]:
# Print best model name
print("Best Model:", best_name)
print("Best Accuracy:", round(best_score * 100, 2), "%")

Best Model: Gradient Boosting
Best Accuracy: 88.68 %


In [12]:
import joblib

# Save best model into pkl file
joblib.dump(best_model, "best_model.pkl")

print("New balanced model saved!")

New balanced model saved!


In [13]:
# Print best model name
print("Best Model:", best_name)

# Print best accuracy
print("Accuracy:", round(best_score * 100, 2), "%")

Best Model: Gradient Boosting
Accuracy: 88.68 %


In [14]:
# Predict again using best model
final_pred = best_model.predict(X_test)

# Show classification report
print(classification_report(y_test, final_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45170
           1       0.62      0.05      0.10      5900

    accuracy                           0.89     51070
   macro avg       0.75      0.52      0.52     51070
weighted avg       0.86      0.89      0.84     51070



In [15]:
# Save best model into pkl file
joblib.dump(best_model, "best_model.pkl")

print("Model saved successfully")

Model saved successfully
